In [0]:
%sql
SELECT tabell, regel,
       count(*)                 AS rows,
       count_if(within_rule)    AS ok,
       max(abs(diff))           AS max_abs_diff,
       max(abs(diff_pct))       AS max_abs_pct,
       min(n_kommuner)          AS min_kommuner,
       max(n_files)             AS max_files
FROM laddstolpar_df.ops.scb_riket_check
GROUP BY ALL
ORDER BY ALL;

In [0]:
%sql
WITH b AS (
  SELECT 'TAB3276' AS tabell, agarkategori AS dim, tid, region, value
  FROM laddstolpar_df.bronze.scb_tab3276
  WHERE agarkategori IN ('000', '010', '020', '030')
  UNION ALL
  SELECT 'TAB3277', drivmedel, tid, region, value
  FROM laddstolpar_df.bronze.scb_tab3277
),
lvl AS (
  SELECT tabell, dim, tid,
         sum(value) FILTER (WHERE region = '00')                                          AS riket,
         sum(value) FILTER (WHERE length(region) = 2 AND region NOT IN ('00','15','16'))  AS sum_lan,
         sum(value) FILTER (WHERE length(region) = 4 AND region <> '1917')                AS sum_kommun
  FROM b
  GROUP BY ALL
)
SELECT tabell, dim, tid, riket, sum_lan, sum_kommun,
       riket   - sum_lan    AS riket_minus_lan,
       sum_lan - sum_kommun AS lan_minus_kommun
FROM lvl
WHERE riket <> sum_kommun
ORDER BY tabell, dim, tid;

In [0]:
%sql
SELECT tabell, regel,
       count(*)               AS rows,
       count_if(within_rule)  AS ok,
       count_if(riket = 0)    AS riket_zero,
       max(abs(diff))         AS max_abs_diff,
       max(abs(diff_pct))     AS max_abs_pct
FROM laddstolpar_df.ops.scb_riket_check
GROUP BY ALL
ORDER BY ALL;

In [0]:
%sql
-- summary trafa
SELECT 'lan_check' AS check_name, count(*) AS rows, count_if(diff = 0) AS exact,
       count_if(lan_total IS NULL) AS missing, max(abs(diff)) AS max_abs_diff
FROM laddstolpar_df.ops.trafa_lan_check
UNION ALL
SELECT 'cross_check', count(*), count_if(diff = 0),
       count_if(trafa_totalt IS NULL OR scb_totalt IS NULL), max(abs(diff))
FROM laddstolpar_df.ops.trafa_scb_cars_check;

In [0]:
%sql
SELECT * FROM laddstolpar_df.ops.scb_retired_code_check ORDER BY tabell, region;